In [6]:
#INSTALACE BALÍČKŮ
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM
#https://huggingface.co/InstaDeepAI/nucleotide-transformer-v2-500m-multi-species
import pandas as pd

In [7]:
#INICIALIZACE MODELU
model_name = "InstaDeepAI/nucleotide-transformer-v2-500m-multi-species"
#případně menší model - zkontroluj dokumentaci

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

model = AutoModelForMaskedLM.from_pretrained(
    model_name,
    trust_remote_code=True
)

model.eval()

c:\Users\sztac\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: c40b5bc7-271c-4c7b-876e-67d887d6343b)')' thrown while requesting HEAD https://huggingface.co/InstaDeepAI/nucleotide-transformer-v2-500m-multi-species/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].


EsmForMaskedLM(
  (esm): EsmModel(
    (embeddings): EsmEmbeddings(
      (word_embeddings): Embedding(4107, 1024, padding_idx=1)
      (dropout): Dropout(p=0.0, inplace=False)
      (position_embeddings): Embedding(2050, 1024, padding_idx=1)
    )
    (encoder): EsmEncoder(
      (layer): ModuleList(
        (0-28): 29 x EsmLayer(
          (attention): EsmAttention(
            (self): EsmSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
              (rotary_embeddings): RotaryEmbedding()
            )
            (output): EsmSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (LayerNorm): LayerNorm((1024,), eps=1e-12

In [8]:
#ZADÁNÍ SEKVENCE
sequence = "ATGCGTACGTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAG"
#zadáš si vlastní sekvence

breakpoint = len(sequence) // 2 

In [9]:
#ZÍSKÁNÍ EMBEDDINGŮ
tokens = tokenizer(
    sequence,
    return_tensors="pt",
    padding="max_length",
    truncation=True,
    max_length=tokenizer.model_max_length
)
#pro následné anlýzy je nutné mít všechny sekvence stejně dlouhé

with torch.no_grad():
    outputs = model(
        tokens["input_ids"],
        attention_mask=tokens["attention_mask"],
        output_hidden_states=True
    )

embeddings = outputs.hidden_states[-1]

breakpoint_token = breakpoint + 1

breakpoint_embedding = embeddings[0, breakpoint_token, :]

In [10]:
#ULOŽENÍ DO SOUBORU
embedding_np = breakpoint_embedding.detach().cpu().numpy()

df = pd.DataFrame(embedding_np.reshape(1, -1))

df.to_csv("breakpoint_embedding.csv", index=False)

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM
#https://huggingface.co/InstaDeepAI/nucleotide-transformer-v2-500m-multi-species
import pandas as pd
%pip uninstall -y transformers accelerate

# Instalace stabilní kombinace pro Nucleotide Transformer
%pip install transformers==4.40.0 accelerate==0.29.0

c:\Users\sztac\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Found existing installation: transformers 4.40.0
Uninstalling transformers-4.40.0:
  Successfully uninstalled transformers-4.40.0
Found existing installation: accelerate 0.29.0
Uninstalling accelerate-0.29.0:
  Successfully uninstalled accelerate-0.29.0
Note: you may need to restart the kernel to use updated packages.
  Using cached transformers-4.40.0-py3-none-any.whl.metadata (137 kB)
  Using cached accelerate-0.29.0-py3-none-any.whl.metadata (18 kB)
Using cached transformers-4.40.0-py3-none-any.whl (9.0 MB)
Using cached accelerate-0.29.0-py3-none-any.whl (297 kB)
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
#Grafika reinstal
import sys
import subprocess

def install_gpu_torch():
    print("--- ZAČÍNÁM OPRAVU PRO GPU (RTX 3060) ---")
    
    # 1. Odinstalace starého Torche
    print("\nKrok 1: Mažu starou verzi (CPU)...")
    subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "torch", "torchvision", "torchaudio", "-y"])
    
    # 2. Instalace správné verze pro tvoji grafiku (CUDA 12.1)
    # Pozor: Tohle stahuje cca 2.5 GB, tak to chvíli potrvá!
    print("\nKrok 2: Stahuji verzi pro tvoji grafiku (cca 2.5 GB)...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", 
        "torch", "torchvision", "torchaudio", 
        "--index-url", "https://download.pytorch.org/whl/cu121"
    ])
    
    print("\n" + "="*40)
    print("HOTOVO! DEJ RESTART KERNEL")
    print("\n" + "="*40)
install_gpu_torch()


--- ZAČÍNÁM OPRAVU PRO GPU (RTX 3060) ---

Krok 1: Mažu starou verzi (CPU)...

Krok 2: Stahuji verzi pro tvoji grafiku (cca 2.5 GB)...

HOTOVO! TEĎ JE TO KRITICKÉ:
1. Podívej se nahoru do lišty VS Code nad tímto kódem.
2. Klikni na tlačítko 'Restart' nebo 'Restart Kernel' (točící se šipka).
3. Teprve pak zkus spustit kód s modelem.


In [1]:
#grafika test
import torch
if torch.cuda.is_available():
    print(f"ÚSPĚCH! Grafika {torch.cuda.get_device_name(0)} je připravena k boji.")
else:
    print("Něco je špatně, stále jedu na CPU.")

ÚSPĚCH! Grafika NVIDIA GeForce RTX 3060 Laptop GPU je připravena k boji.


In [4]:
#START
import torch
import pandas as pd
import numpy as np
import sys
from transformers import AutoTokenizer, AutoModel  # Tohle ti chybělo!
from torch.nn.functional import cosine_similarity

# Nastavení zařízení na GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Používám: {device}")
if torch.cuda.is_available():
    print(f"Grafická karta: {torch.cuda.get_device_name(0)}")

from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch

model_name = "InstaDeepAI/nucleotide-transformer-v2-500m-multi-species"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Načtení tokenizeru
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Načtení modelu v plné přesnosti (float32)
model = AutoModelForMaskedLM.from_pretrained(
    model_name, 
    trust_remote_code=True
    # torch_dtype je pryč, default je float32
)

model = model.to(device)
model.eval()

print(f"Model je načten na zařízení: {device}")

c:\Users\sztac\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Používám: cuda
Grafická karta: NVIDIA GeForce RTX 3060 Laptop GPU


c:\Users\sztac\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Model je načten na zařízení: cuda


In [21]:
#Nacteni sekvence a BP PO 1200 krocich
full_sequence = """TTCCAGGACTGCAGAACTGGCCCAGACCTCTGTATTGGAAAGGTCTTTATGGACCAGGGAGTCCGGTGTCTTTTTTACGGGGGACCCCTG
GGCTGCGAGTTGCACAGTCCAATTCGCTGTTGTTAGGGCCTCAGTTTCCCAAAAGGCACAGGGACGGGGGGAGGGTGGCGGCTCGATGGG
GGAGCCGCCTCCAGGGGGCCCCCCCGCCCTGTGCCCACGGCGCGGCCCCTTTAAGAGGCCCGCCTGGCTCCGTCATCCGCGCCGCGGCCA
CCTCCCCCCGGCCCTCCCCTTCCTGCGGCGCAGAGTGCGGGCCGGGCGGGAGTGCGGCGAGAGCCGGCTGGCTGAGCTTAGCGTCCGAGG
AGGCGGCGGCGGCGGCGGCGGCACGGCGGCGGCGGGGCTGTGGGGCGGTGCGGAAGCGAGAGGCGAGGAGCGCGCGGGCCGTGGCCAGAG
TCTGGCGGCGGCCTGGCGGAGCGGAGAGCAGCGCCCGCGCCTCGCCGTGCGGAGGAGCCCCGCACACAATAGCGGCGCGCGCAGCCCGCG
CCCTTCCCCCCGGCGCGCCCCGCCCCGCGCGCCGAGCGCCCCGCTCCGCCTCACCTGCCACCAGGGAGTGGGCGGGCATTGTTCGCCGCC
GCCGCCGCCGCGCGGGCCATGGGGGCCGCCCGGCGCCCGGGGCCGGGCTGGCGAGGCGCCGCGCCGCCGCTGAGACGGGCCCCGCGCGCA
GCCCGGCGGCGCAGGTAAGGCCGGCCGCGCCATGGTGGACCCGGTGGGCTTCGCGGAGGCGTGGAAGGCGCAGTTCCCGGACTCAGAGCC
CCCGCGCATGGAGCTGCGCTCAGTGGGCGACATCGAGCAGGAGCTGGAGCGCTGCAAGGCCTCCATTCGGCGCCTGGAGCAGGAGGTGAA
CCAGGAGCGCTTCCGCATGATCTACCTGCAGACGTTGCTGGCCAAGGAAAAGAAGAGCTATGACCGGCAGCGATGGGGCTTCCGGCGCGC
GGCGCAGGCCCCCGACGGCGCCTCCGAGCCCCGAGCGTCCGCGTCGCGCCCGCAGCCAGCGCCCGCCGACGGAGCCGACCCGCCGCCCGC
CGAGGAGCCCGAGGCCCGGCCCGACGGCGAGGGTTCTCCGGGTAAGGCCAGGCCCGGGACCGCCCGCAGGCCCGGGGCAGCCGCGTCGGG
GGAACGGGACGACCGGGGACCCCCCGCCAGCGTGGCGGCGCTCAGGTCCAACTTCGAGCGGATCCGCAAGGGCCATGGCCAGCCCGGGGC
GGACGCCGAGAAGCCCTTCTACGTGAACGTCGAGTTTCACCACGAGCGCGGCCTGGTGAAGGTCAACGACAAAGAGGTGTCGGACCGCAT
CAGCTCCCTGGGCAGCCAGGCCATGCAGATGGAGCGCAAAAAGTCCCAGCACGGCGCGGGCTCGAGCGTGGGGGATGCATCCAGGCCCCC
TTACCGGGGACGCTCCTCGGAGAGCAGCTGCGGCGTCGACGGCGACTACGAGGACGCCGAGTTGAACCCCCGCTTCCTGAAGGACAACCT
GATCGACGCCAATGGCGGTAGCAGGCCCCCTTGGCCGCCCCTGGAGTACCAGCCCTACCAGAGCATCTACGTCGGGGGCATGATGGAAGG
GGAGGGCAAGGGCCCGCTCCTGCGCAGCCAGAGCACCTCTGAGCAGGAGAAGCGCCTTACCTGGCCCCGCAGGTCCTACTCCCCCCGGAG
TTTTGAGGATTGCGGAGGCGGCTATACCCCGGACTGCAGCTCCAATGAGAACCTCACCTCCAGCGAGGAGGACTTCTCCTCTGGCCAGTC
CAGCCGCGTGTCCCCAAGCCCCACCACCTACCGCATGTTCCGGGACAAAAGCCGCTCTCCCTCGCAGAACTCGCAACAGTCCTTCGACAG
CAGCAGTCCCCCCACGCCGCAGTGCCATAAGCGGCACCGGCACTGCCCGGTTGTCGTGTCCGAGGCCACCATCGTGGGCGTCCGCAAGAC
CGGGCAGATCTGGCCCAACGATGGCGAGGGCGCCTTCCATGGAGACGCAGATGGCTCGTTCGGAACACCACCTGGATACGGCTGCGCTGC
AGACCGGGCAGAGGAGCAGCGCCGGCACCAAGATGGGCTGCCCTACATTGATGACTCGCCCTCCTCATCGCCCCACCTCAGCAGCAAGGG
CAGGGGCAGCCGGGATGCGCTGGTCTCGGGAGCCCTGGAGTCCACTAAAGCGAGTGAGCTGGACTTGGAAAAGGGCTTGGAGATGAGAAA
ATGGGTCCTGTCGGGAATCCTGGCTAGCGAGGAGACTTACCTGAGCCACCTGGAGGCACTGCTGCTGCCCATGAAGCCTTTGAAAGCCGC
TGCCACCACCTCTCAGCCGGTGCTGACGAGTCAGCAGATCGAGACCATCTTCTTCAAAGTGCCTGAGCTCTACGAGATCCACAAGGAGTT
CTATGATGGGCTCTTCCCCCGCGTGCAGCAGTGGAGCCACCAGCAGCGGGTGGGCGACCTCTTCCAGAAGCTGGCCAGCCAGCTGGGTGT
GTACCGGGCCTTCGTGGACAACTACGGAGTTGCCATGGAAATGGCTGAGAAGTGCTGTCAGGCCAATGCTCAGTTTGCAGAAATCTCCGA
GAACCTGAGAGCCAGAAGCAACAAAGATGCCAAGGATCCAACGACCAAGAACTCTCTGGAAACTCTGCTCTACAAGCCTGTGGACCGTGT
GACGAGGAGCACGCTGGTCCTCCATGACTTGCTGAAGCACACTCCTGCCAGCCACCCTGACCACCCCTTGCTGCAGGACGCCCTCCGCAT
CTCACAGAACTTCCTGTCCAGCATCAATGAGGAGATCACACCCCGACGGCAGTCCATGACGGTGAAGAAGGGAGAGCACCGGCAGCTGCT
GAAGGACAGCTTCATGGTGGAGCTGGTGGAGGGGGCCCGCAAGCTGCGCCACGTCTTCCTGTTCACCGACCTGCTTCTCTGCACCAAGCT
CAAGAAGCAGAGCGGAGGCAAAACGCAGCAGTATGACTGCAAATGGTACATTCCGCTCACGGATCTCAGCTTCCAGATGGTGGATGAACT
GGAGGCAGTGCCCAACATCCCCCTGGTGCCCGATGAGGAGCTGGACGCTTTGAAGATCAAGATCTCCCAGATCAAGAATGACATCCAGAG
AGAGAAGAGGGCGAACAAGGGCAGCAAGGCTACGGAGAGGCTGAAGAAGAAGCTGTCGGAGCAGGAGTCACTGCTGCTGCTTATGTCTCC
CAGCATGGCCTTCAGGGTGCACAGCCGCAACGGCAAGAGTTACACGTTCCTGATCTCCTCTGACTATGAGCGTGCAGAGTGGAGGGAGAA
CATCCGGGAGCAGCAGAAGAAGTGTTTCAGAAGCTTCTCCCTGACATCCGTGGAGCTGCAGATGCTGACCAACTCGTGTGTGAAACTCCA
GACTGTCCACAGCATTCCGCTGACCATCAATAAGGAAGATGATGAGTCTCCGGGGCTCTATGGGTTTCTGAATGTCATCGTCCACTCAGC
CACTGGATTTAAGCAGAGTTCAAATCTGTACTGCACCCTGGAGGTGGATTCCTTTGGGTATTTTGTGAATAAAGCAAAGACGCGCGTCTA
CAGGGACACAGCTGAGCCAAACTGGAACGAGGAATTTGAGATAGAGCTGGAGGGCTCCCAGACCCTGAGGATACTGTGCTATGAAAAGTG
TTACAACAAGACGAAGATCCCCAAGGAGGACGGCGAGAGCACGGACAGACTCATGGGGAAGGGCCAGGTCCAGCTGGACCCGCAGGCCCT
GCAGGACAGAGACTGGCAGCGCACCGTCATCGCCATGAATGGGATCGAAGTAAAGCTCTCGGTCAAGTTCAACAGCAGGGAGTTCAGCTT
GAAGAGGATGCCGTCCCGAAAACAGACAGGGGTCTTCGGAGTCAAGATTGCTGTGGTCACCAAGAGAGAGAGGTCCAAGGTGCCCTACAT
CGTGCGCCAGTGCGTGGAGGAGATCGAGCGCCGAGGCATGGAGGAGGTGGGCATCTACCGCGTGTCCGGTGTGGCCACGGACATCCAGGC
ACTGAAGGCAGCCTTCGACGTCAAAGCCCTTCAGCGGCCAGTAGCATCTGACTTTGAGCCTCAGGGTCTGAGTGAAGCCGCTCGTTGGAA
CTCCAAGGAAAACCTTCTCGCTGGACCCAGTGAAAATGACCCCAACCTTTTCGTTGCACTGTATGATTTTGTGGCCAGTGGAGATAACAC
TCTAAGCATAACTAAAGGTGAAAAGCTCCGGGTCTTAGGCTATAATCACAATGGGGAATGGTGTGAAGCCCAAACCAAAAATGGCCAAGG
CTGGGTCCCAAGCAACTACATCACGCCAGTCAACAGTCTGGAGAAACACTCCTGGTACCATGGGCCTGTGTCCCGCAATGCCGCTGAGTA
TCTGCTGAGCAGCGGGATCAATGGCAGCTTCTTGGTGCGTGAGAGTGAGAGCAGTCCTGGCCAGAGGTCCATCTCGCTGAGATACGAAGG
GAGGGTGTACCATTACAGGATCAACACTGCTTCTGATGGCAAGCTCTACGTCTCCTCCGAGAGCCGCTTCAACACCCTGGCCGAGTTGGT
TCATCATCATTCAACGGTGGCCGACGGGCTCATCACCACGCTCCATTATCCAGCCCCAAAGCGCAACAAGCCCACTGTCTATGGTGTGTC
CCCCAACTACGACAAGTGGGAGATGGAACGCACGGACATCACCATGAAGCACAAGCTGGGCGGGGGCCAGTACGGGGAGGTGTACGAGGG
CGTGTGGAAGAAATACAGCCTGACGGTGGCCGTGAAGACCTTGAAGGAGGACACCATGGAGGTGGAAGAGTTCTTGAAAGAAGCTGCAGT
CATGAAAGAGATCAAACACCCTAACCTGGTGCAGCTCCTTGGGGTCTGCACCCGGGAGCCCCCGTTCTATATCATCACTGAGTTCATGAC
CTACGGGAACCTCCTGGACTACCTGAGGGAGTGCAACCGGCAGGAGGTGAACGCCGTGGTGCTGCTGTACATGGCCACTCAGATCTCGTC
AGCCATGGAGTACCTGGAGAAGAAAAACTTCATCCACAGAGATCTTGCTGCCCGAAACTGCCTGGTAGGGGAGAACCACTTGGTGAAGGT
AGCTGATTTTGGCCTGAGCAGGTTGATGACAGGGGACACCTACACAGCCCATGCTGGAGCCAAGTTCCCCATCAAATGGACTGCACCCGA
GAGCCTGGCCTACAACAAGTTCTCCATCAAGTCCGACGTCTGGGCATTTGGAGTATTGCTTTGGGAAATTGCTACCTATGGCATGTCCCC
TTACCCGGGAATTGACCTGTCCCAGGTGTATGAGCTGCTAGAGAAGGACTACCGCATGGAGCGCCCAGAAGGCTGCCCAGAGAAGGTCTA
TGAACTCATGCGAGCATGTTGGCAGTGGAATCCCTCTGACCGGCCCTCCTTTGCTGAAATCCACCAAGCCTTTGAAACAATGTTCCAGGA
ATCCAGTATCTCAGACGAAGTGGAAAAGGAGCTGGGGAAACAAGGCGTCCGTGGGGCTGTGAGTACCTTGCTGCAGGCCCCAGAGCTGCC
CACCAAGACGAGGACCTCCAGGAGAGCTGCAGAGCACAGAGACACCACTGACGTGCCTGAGATGCCTCACTCCAAGGGCCAGGGAGAGAG
CGATCCTCTGGACCATGAGCCTGCCGTGTCTCCATTGCTCCCTCGAAAAGAGCGAGGTCCCCCGGAGGGCGGCCTGAATGAAGATGAGCG
CCTTCTCCCCAAAGACAAAAAGACCAACTTGTTCAGCGCCTTGATCAAGAAGAAGAAGAAGACAGCCCCAACCCCTCCCAAACGCAGCAG
CTCCTTCCGGGAGATGGACGGCCAGCCGGAGCGCAGAGGGGCCGGCGAGGAAGAGGGCCGAGACATCAGCAACGGGGCACTGGCTTTCAC
CCCCTTGGACACAGCTGACCCAGCCAAGTCCCCAAAGCCCAGCAATGGGGCTGGGGTCCCCAATGGAGCCCTCCGGGAGTCCGGGGGCTC
AGGCTTCCGGTCTCCCCACCTGTGGAAGAAGTCCAGCACGCTGACCAGCAGCCGCCTAGCCACCGGCGAGGAGGAGGGCGGTGGCAGCTC
CAGCAAGCGCTTCCTGCGCTCTTGCTCCGCCTCCTGCGTTCCCCATGGGGCCAAGGACACGGAGTGGAGGTCAGTCACGCTGCCTCGGGA
CTTGCAGTCCACGGGAAGACAGTTTGACTCGTCCACATTTGGAGGGCACAAAAGTGAGAAGCCGGCTCTGCCTCGGAAGAGGGCAGGGGA
GAACAGGTCTGACCAGGTGACCCGAGGCACAGTAACGCCTCCCCCCAGGCTGGTGAAAAAGAATGAGGAAGCTGCTGATGAGGTCTTCAA
AGACATCATGGAGTCCAGCCCGGGCTCCAGCCCGCCCAACCTGACTCCAAAACCCCTCCGGCGGCAGGTCACCGTGGCCCCTGCCTCGGG
CCTCCCCCACAAGGAAGAAGCTGGAAAGGGCAGTGCCTTAGGGACCCCTGCTGCAGCTGAGCCAGTGACCCCCACCAGCAAAGCAGGCTC
AGGTGCACCAGGGGGCACCAGCAAGGGCCCCGCCGAGGAGTCCAGAGTGAGGAGGCACAAGCACTCCTCTGAGTCGCCAGGGAGGGACAA
GGGGAAATTGTCCAGGCTCAAACCTGCCCCGCCGCCCCCACCAGCAGCCTCTGCAGGGAAGGCTGGAGGAAAGCCCTCGCAGAGCCCGAG
CCAGGAGGCGGCCGGGGAGGCAGTCCTGGGCGCAAAGACAAAAGCCACGAGTCTGGTTGATGCTGTGAACAGTGACGCTGCCAAGCCCAG
CCAGCCGGGAGAGGGCCTCAAAAAGCCCGTGCTCCCGGCCACTCCAAAGCCACAGTCCGCCAAGCCGTCGGGGACCCCCATCAGCCCAGC
CCCCGTTCCCTCCACGTTGCCATCAGCATCCTCGGCCCTGGCAGGGGACCAGCCGTCTTCCACCGCCTTCATCCCTCTCATATCAACCCG
AGTGTCTCTTCGGAAAACCCGCCAGCCTCCAGAGCGGATCGCCAGCGGCGCCATCACCAAGGGCGTGGTCCTGGACAGCACCGAGGCGCT
GTGCCTCGCCATCTCTAGGAACTCCGAGCAGATGGCCAGCCACAGCGCAGTGCTGGAGGCCGGCAAAAACCTCTACACGTTCTGCGTGAG
CTATGTGGATTCCATCCAGCAAATGAGGAACAAGTTTGCCTTCCGAGAGGCCATCAACAAACTGGAGAATAATCTCCGGGAGCTTCAGAT
CTGCCCGGCGACAGCAGGCAGTGGTCCAGCGGCCACTCAGGACTTCAGCAAGCTCCTCAGTTCGGTGAAGGAAATCAGTGACATAGTGCA
GAGGTAGCAGCAGTCAGGGGTCAGGTGTCAGGCCCGTCGGAGCTGCCTGCAGCACATGCGGGCTCGCCCATACCCGTGACAGTGGCTGAC
AAGGGACTAGTGAGTCAGCACCTTGGCCCAGGAGCTCTGCGCCAGGCAGAGCTGAGGGCCCTGTGGAGTCCAGCTCTACTACCTACGTTT
GCACCGCCTGCCCTCCCGCACCTTCCTCCTCCCCGCTCCGTCTCTGTCCTCGAATTTTATCTGTGGAGTTCCTGCTCCGTGGACTGCAGT
CGGCATGCCAGGACCCGCCAGCCCCGCTCCCACCTAGTGCCCCAGACTGAGCTCTCCAGGCCAGGTGGGAACGGCTGATGTGGACTGTCT
TTTTCATTTTTTTCTCTCTGGAGCCCCTCCTCCCCCGGCTGGGCCTCCTTCTTCCACTTCTCCAAGAATGGAAGCCTGAACTGAGGCCTT
GTGTGTCAGGCCCTCTGCCTGCACTCCCTGGCCTTGCCCGTCGTGTGCTGAAGACATGTTTCAAGAACCGCATTTCGGGAAGGGCATGCA
CGGGCATGCACACGGCTGGTCACTCTGCCCTCTGCTGCTGCCCGGGGTGGGGTGCACTCGCCATTTCCTCACGTGCAGGACAGCTCTTGA
TTTGGGTGGAAAACAGGGTGCTAAAGCCAACCAGCCTTTGGGTCCTGGGCAGGTGGGAGCTGAAAAGGATCGAGGCATGGGGCATGTCCT
TTCCATCTGTCCACATCCCCAGAGCCCAGCTCTTGCTCTCTTGTGACGTGCACTGTGAATCCTGGCAAGAAAGCTTGAGTCTCAAGGGTG
GCAGGTCACTGTCACTGCCGACATCCCTCCCCCAGCAGAATGGAGGCAGGGGACAAGGGAGGCAGTGGCTAGTGGGGTGAACAGCTGGTG
CCAAATAGCCCCAGACTGGGCCCAGGCAGGTCTGCAAGGGCCCAGAGTGAACCGTCCTTTCACACATCTGGGTGCCCTGAAAGGGCCCTT
CCCCTCCCCCACTCCTCTAAGACAAAGTAGATTCTTACAAGGCCCTTTCCTTTGGAACAAGACAGCCTTCACTTTTCTGAGTTCTTGAAG
CATTTCAAAGCCCTGCCTCTGTGTAGCCGCCCTGAGAGAGAATAGAGCTGCCACTGGGCACCTGCGCACAGGTGGGAGGAAAGGGCCTGG
CCAGTCCTGGTCCTGGCTGCACTCTTGAACTGGGCGAATGTCTTATTTAATTACCGTGAGTGACATAGCCTCATGTTCTGTGGGGGTCAT
CAGGGAGGGTTAGGAAAACCACAAACGGAGCCCCTGAAAGCCTCACGTATTTCACAGAGCACGCCTGCCATCTTCTCCCCGAGGCTGCCC
CAGGCCGGAGCCCAGATACGGGGGCTGTGACTCTGGGCAGGGACCCGGGGTCTCCTGGACCTTGACAGAGCAGCTAACTCCGAGAGCAGT
GGGCAGGTGGCCGCCCCTGAGGCTTCACGCCGGGAGAAGCCACCTTCCCACCCCTTCATACCGCCTCGTGCCAGCAGCCTCGCACAGGCC
CTAGCTTTACGCTCATCACCTAAACTTGTACTTTATTTTTCTGATAGAAATGGTTTCCTCTGGATCGTTTTATGCGGTTCTTACAGCACA
TCACCTCTTTGCCCCCGACGGCTGTGACGCAGCCGGAGGGAGGCACTAGTCACCGACAGCGGCCTTGAAGACAGAGCAAAGCGCCCACCC
AGGTCCCCCGACTGCCTGTCTCCATGAGGTACTGGTCCCTTCCTTTTGTTAACGTGATGTGCCACTATATTTTACACGTATCTCTTGGTA
TGCATCTTTTATAGACGCTCTTTTCTAAGTGGCGTGTGCATAGCGTCCTGCCCTGCCCCCTCGGGGGCCTGTGGTGGCTCCCCCTCTGCT
TCTCGGGGTCCAGTGCATTTTGTTTCTGTATATGATTCTCTGTGGTTTTTTTTGAATCCAAATCTGTCCTCTGTAGTATTTTTTAAATAA
ATCAGTGTTTACATTAGAA""".replace("\n", "").replace(" ", "")

bp_index = 4073
# 1. Výpočet maximálního symetrického okna (dělitelného 12)
vzdalenost_k_zacatku = bp_index
vzdalenost_ke_konci = len(full_sequence) - bp_index

max_mozne = min(vzdalenost_k_zacatku, vzdalenost_ke_konci) * 2
max_symetricke_12 = (max_mozne // 12) * 12

print(f"Tvůj zlom: {bp_index}")
print(f"Maximální okno (symetrické): {max_symetricke_12} bp")

# 2. Generování oken po násobcích 1200
window_sizes = []
krok = 1200
aktualni_velikost = 120 # Začneme malým detailem pro srovnání

while aktualni_velikost < max_symetricke_12:
    window_sizes.append(aktualni_velikost)
    
    # První skok uděláme na 1200, pak už jdeme po 1200
    if aktualni_velikost == 120:
        aktualni_velikost = 1200
    else:
        aktualni_velikost += 1200

# 3. Přidání největšího okna
if max_symetricke_12 not in window_sizes and max_symetricke_12 >= 120:
    window_sizes.append(max_symetricke_12)

window_sizes.sort()

print(f"Vygenerovaná lineární okna (krok 1200): {window_sizes}")
print(f"Max okno dělitelné 12: {max_symetricke_12 % 12 == 0}")

Tvůj zlom: 4073
Maximální okno (symetrické): 8136 bp
Vygenerovaná lineární okna (krok 1200): [120, 1200, 2400, 3600, 4800, 6000, 7200, 8136]
Max okno dělitelné 12: True


In [22]:
#Nacteni sekvence a BP Exponencialni kroky
full_sequence = """TTCCAGGACTGCAGAACTGGCCCAGACCTCTGTATTGGAAAGGTCTTTATGGACCAGGGAGTCCGGTGTCTTTTTTACGGGGGACCCCTG
GGCTGCGAGTTGCACAGTCCAATTCGCTGTTGTTAGGGCCTCAGTTTCCCAAAAGGCACAGGGACGGGGGGAGGGTGGCGGCTCGATGGG
GGAGCCGCCTCCAGGGGGCCCCCCCGCCCTGTGCCCACGGCGCGGCCCCTTTAAGAGGCCCGCCTGGCTCCGTCATCCGCGCCGCGGCCA
CCTCCCCCCGGCCCTCCCCTTCCTGCGGCGCAGAGTGCGGGCCGGGCGGGAGTGCGGCGAGAGCCGGCTGGCTGAGCTTAGCGTCCGAGG
AGGCGGCGGCGGCGGCGGCGGCACGGCGGCGGCGGGGCTGTGGGGCGGTGCGGAAGCGAGAGGCGAGGAGCGCGCGGGCCGTGGCCAGAG
TCTGGCGGCGGCCTGGCGGAGCGGAGAGCAGCGCCCGCGCCTCGCCGTGCGGAGGAGCCCCGCACACAATAGCGGCGCGCGCAGCCCGCG
CCCTTCCCCCCGGCGCGCCCCGCCCCGCGCGCCGAGCGCCCCGCTCCGCCTCACCTGCCACCAGGGAGTGGGCGGGCATTGTTCGCCGCC
GCCGCCGCCGCGCGGGCCATGGGGGCCGCCCGGCGCCCGGGGCCGGGCTGGCGAGGCGCCGCGCCGCCGCTGAGACGGGCCCCGCGCGCA
GCCCGGCGGCGCAGGTAAGGCCGGCCGCGCCATGGTGGACCCGGTGGGCTTCGCGGAGGCGTGGAAGGCGCAGTTCCCGGACTCAGAGCC
CCCGCGCATGGAGCTGCGCTCAGTGGGCGACATCGAGCAGGAGCTGGAGCGCTGCAAGGCCTCCATTCGGCGCCTGGAGCAGGAGGTGAA
CCAGGAGCGCTTCCGCATGATCTACCTGCAGACGTTGCTGGCCAAGGAAAAGAAGAGCTATGACCGGCAGCGATGGGGCTTCCGGCGCGC
GGCGCAGGCCCCCGACGGCGCCTCCGAGCCCCGAGCGTCCGCGTCGCGCCCGCAGCCAGCGCCCGCCGACGGAGCCGACCCGCCGCCCGC
CGAGGAGCCCGAGGCCCGGCCCGACGGCGAGGGTTCTCCGGGTAAGGCCAGGCCCGGGACCGCCCGCAGGCCCGGGGCAGCCGCGTCGGG
GGAACGGGACGACCGGGGACCCCCCGCCAGCGTGGCGGCGCTCAGGTCCAACTTCGAGCGGATCCGCAAGGGCCATGGCCAGCCCGGGGC
GGACGCCGAGAAGCCCTTCTACGTGAACGTCGAGTTTCACCACGAGCGCGGCCTGGTGAAGGTCAACGACAAAGAGGTGTCGGACCGCAT
CAGCTCCCTGGGCAGCCAGGCCATGCAGATGGAGCGCAAAAAGTCCCAGCACGGCGCGGGCTCGAGCGTGGGGGATGCATCCAGGCCCCC
TTACCGGGGACGCTCCTCGGAGAGCAGCTGCGGCGTCGACGGCGACTACGAGGACGCCGAGTTGAACCCCCGCTTCCTGAAGGACAACCT
GATCGACGCCAATGGCGGTAGCAGGCCCCCTTGGCCGCCCCTGGAGTACCAGCCCTACCAGAGCATCTACGTCGGGGGCATGATGGAAGG
GGAGGGCAAGGGCCCGCTCCTGCGCAGCCAGAGCACCTCTGAGCAGGAGAAGCGCCTTACCTGGCCCCGCAGGTCCTACTCCCCCCGGAG
TTTTGAGGATTGCGGAGGCGGCTATACCCCGGACTGCAGCTCCAATGAGAACCTCACCTCCAGCGAGGAGGACTTCTCCTCTGGCCAGTC
CAGCCGCGTGTCCCCAAGCCCCACCACCTACCGCATGTTCCGGGACAAAAGCCGCTCTCCCTCGCAGAACTCGCAACAGTCCTTCGACAG
CAGCAGTCCCCCCACGCCGCAGTGCCATAAGCGGCACCGGCACTGCCCGGTTGTCGTGTCCGAGGCCACCATCGTGGGCGTCCGCAAGAC
CGGGCAGATCTGGCCCAACGATGGCGAGGGCGCCTTCCATGGAGACGCAGATGGCTCGTTCGGAACACCACCTGGATACGGCTGCGCTGC
AGACCGGGCAGAGGAGCAGCGCCGGCACCAAGATGGGCTGCCCTACATTGATGACTCGCCCTCCTCATCGCCCCACCTCAGCAGCAAGGG
CAGGGGCAGCCGGGATGCGCTGGTCTCGGGAGCCCTGGAGTCCACTAAAGCGAGTGAGCTGGACTTGGAAAAGGGCTTGGAGATGAGAAA
ATGGGTCCTGTCGGGAATCCTGGCTAGCGAGGAGACTTACCTGAGCCACCTGGAGGCACTGCTGCTGCCCATGAAGCCTTTGAAAGCCGC
TGCCACCACCTCTCAGCCGGTGCTGACGAGTCAGCAGATCGAGACCATCTTCTTCAAAGTGCCTGAGCTCTACGAGATCCACAAGGAGTT
CTATGATGGGCTCTTCCCCCGCGTGCAGCAGTGGAGCCACCAGCAGCGGGTGGGCGACCTCTTCCAGAAGCTGGCCAGCCAGCTGGGTGT
GTACCGGGCCTTCGTGGACAACTACGGAGTTGCCATGGAAATGGCTGAGAAGTGCTGTCAGGCCAATGCTCAGTTTGCAGAAATCTCCGA
GAACCTGAGAGCCAGAAGCAACAAAGATGCCAAGGATCCAACGACCAAGAACTCTCTGGAAACTCTGCTCTACAAGCCTGTGGACCGTGT
GACGAGGAGCACGCTGGTCCTCCATGACTTGCTGAAGCACACTCCTGCCAGCCACCCTGACCACCCCTTGCTGCAGGACGCCCTCCGCAT
CTCACAGAACTTCCTGTCCAGCATCAATGAGGAGATCACACCCCGACGGCAGTCCATGACGGTGAAGAAGGGAGAGCACCGGCAGCTGCT
GAAGGACAGCTTCATGGTGGAGCTGGTGGAGGGGGCCCGCAAGCTGCGCCACGTCTTCCTGTTCACCGACCTGCTTCTCTGCACCAAGCT
CAAGAAGCAGAGCGGAGGCAAAACGCAGCAGTATGACTGCAAATGGTACATTCCGCTCACGGATCTCAGCTTCCAGATGGTGGATGAACT
GGAGGCAGTGCCCAACATCCCCCTGGTGCCCGATGAGGAGCTGGACGCTTTGAAGATCAAGATCTCCCAGATCAAGAATGACATCCAGAG
AGAGAAGAGGGCGAACAAGGGCAGCAAGGCTACGGAGAGGCTGAAGAAGAAGCTGTCGGAGCAGGAGTCACTGCTGCTGCTTATGTCTCC
CAGCATGGCCTTCAGGGTGCACAGCCGCAACGGCAAGAGTTACACGTTCCTGATCTCCTCTGACTATGAGCGTGCAGAGTGGAGGGAGAA
CATCCGGGAGCAGCAGAAGAAGTGTTTCAGAAGCTTCTCCCTGACATCCGTGGAGCTGCAGATGCTGACCAACTCGTGTGTGAAACTCCA
GACTGTCCACAGCATTCCGCTGACCATCAATAAGGAAGATGATGAGTCTCCGGGGCTCTATGGGTTTCTGAATGTCATCGTCCACTCAGC
CACTGGATTTAAGCAGAGTTCAAATCTGTACTGCACCCTGGAGGTGGATTCCTTTGGGTATTTTGTGAATAAAGCAAAGACGCGCGTCTA
CAGGGACACAGCTGAGCCAAACTGGAACGAGGAATTTGAGATAGAGCTGGAGGGCTCCCAGACCCTGAGGATACTGTGCTATGAAAAGTG
TTACAACAAGACGAAGATCCCCAAGGAGGACGGCGAGAGCACGGACAGACTCATGGGGAAGGGCCAGGTCCAGCTGGACCCGCAGGCCCT
GCAGGACAGAGACTGGCAGCGCACCGTCATCGCCATGAATGGGATCGAAGTAAAGCTCTCGGTCAAGTTCAACAGCAGGGAGTTCAGCTT
GAAGAGGATGCCGTCCCGAAAACAGACAGGGGTCTTCGGAGTCAAGATTGCTGTGGTCACCAAGAGAGAGAGGTCCAAGGTGCCCTACAT
CGTGCGCCAGTGCGTGGAGGAGATCGAGCGCCGAGGCATGGAGGAGGTGGGCATCTACCGCGTGTCCGGTGTGGCCACGGACATCCAGGC
ACTGAAGGCAGCCTTCGACGTCAAAGCCCTTCAGCGGCCAGTAGCATCTGACTTTGAGCCTCAGGGTCTGAGTGAAGCCGCTCGTTGGAA
CTCCAAGGAAAACCTTCTCGCTGGACCCAGTGAAAATGACCCCAACCTTTTCGTTGCACTGTATGATTTTGTGGCCAGTGGAGATAACAC
TCTAAGCATAACTAAAGGTGAAAAGCTCCGGGTCTTAGGCTATAATCACAATGGGGAATGGTGTGAAGCCCAAACCAAAAATGGCCAAGG
CTGGGTCCCAAGCAACTACATCACGCCAGTCAACAGTCTGGAGAAACACTCCTGGTACCATGGGCCTGTGTCCCGCAATGCCGCTGAGTA
TCTGCTGAGCAGCGGGATCAATGGCAGCTTCTTGGTGCGTGAGAGTGAGAGCAGTCCTGGCCAGAGGTCCATCTCGCTGAGATACGAAGG
GAGGGTGTACCATTACAGGATCAACACTGCTTCTGATGGCAAGCTCTACGTCTCCTCCGAGAGCCGCTTCAACACCCTGGCCGAGTTGGT
TCATCATCATTCAACGGTGGCCGACGGGCTCATCACCACGCTCCATTATCCAGCCCCAAAGCGCAACAAGCCCACTGTCTATGGTGTGTC
CCCCAACTACGACAAGTGGGAGATGGAACGCACGGACATCACCATGAAGCACAAGCTGGGCGGGGGCCAGTACGGGGAGGTGTACGAGGG
CGTGTGGAAGAAATACAGCCTGACGGTGGCCGTGAAGACCTTGAAGGAGGACACCATGGAGGTGGAAGAGTTCTTGAAAGAAGCTGCAGT
CATGAAAGAGATCAAACACCCTAACCTGGTGCAGCTCCTTGGGGTCTGCACCCGGGAGCCCCCGTTCTATATCATCACTGAGTTCATGAC
CTACGGGAACCTCCTGGACTACCTGAGGGAGTGCAACCGGCAGGAGGTGAACGCCGTGGTGCTGCTGTACATGGCCACTCAGATCTCGTC
AGCCATGGAGTACCTGGAGAAGAAAAACTTCATCCACAGAGATCTTGCTGCCCGAAACTGCCTGGTAGGGGAGAACCACTTGGTGAAGGT
AGCTGATTTTGGCCTGAGCAGGTTGATGACAGGGGACACCTACACAGCCCATGCTGGAGCCAAGTTCCCCATCAAATGGACTGCACCCGA
GAGCCTGGCCTACAACAAGTTCTCCATCAAGTCCGACGTCTGGGCATTTGGAGTATTGCTTTGGGAAATTGCTACCTATGGCATGTCCCC
TTACCCGGGAATTGACCTGTCCCAGGTGTATGAGCTGCTAGAGAAGGACTACCGCATGGAGCGCCCAGAAGGCTGCCCAGAGAAGGTCTA
TGAACTCATGCGAGCATGTTGGCAGTGGAATCCCTCTGACCGGCCCTCCTTTGCTGAAATCCACCAAGCCTTTGAAACAATGTTCCAGGA
ATCCAGTATCTCAGACGAAGTGGAAAAGGAGCTGGGGAAACAAGGCGTCCGTGGGGCTGTGAGTACCTTGCTGCAGGCCCCAGAGCTGCC
CACCAAGACGAGGACCTCCAGGAGAGCTGCAGAGCACAGAGACACCACTGACGTGCCTGAGATGCCTCACTCCAAGGGCCAGGGAGAGAG
CGATCCTCTGGACCATGAGCCTGCCGTGTCTCCATTGCTCCCTCGAAAAGAGCGAGGTCCCCCGGAGGGCGGCCTGAATGAAGATGAGCG
CCTTCTCCCCAAAGACAAAAAGACCAACTTGTTCAGCGCCTTGATCAAGAAGAAGAAGAAGACAGCCCCAACCCCTCCCAAACGCAGCAG
CTCCTTCCGGGAGATGGACGGCCAGCCGGAGCGCAGAGGGGCCGGCGAGGAAGAGGGCCGAGACATCAGCAACGGGGCACTGGCTTTCAC
CCCCTTGGACACAGCTGACCCAGCCAAGTCCCCAAAGCCCAGCAATGGGGCTGGGGTCCCCAATGGAGCCCTCCGGGAGTCCGGGGGCTC
AGGCTTCCGGTCTCCCCACCTGTGGAAGAAGTCCAGCACGCTGACCAGCAGCCGCCTAGCCACCGGCGAGGAGGAGGGCGGTGGCAGCTC
CAGCAAGCGCTTCCTGCGCTCTTGCTCCGCCTCCTGCGTTCCCCATGGGGCCAAGGACACGGAGTGGAGGTCAGTCACGCTGCCTCGGGA
CTTGCAGTCCACGGGAAGACAGTTTGACTCGTCCACATTTGGAGGGCACAAAAGTGAGAAGCCGGCTCTGCCTCGGAAGAGGGCAGGGGA
GAACAGGTCTGACCAGGTGACCCGAGGCACAGTAACGCCTCCCCCCAGGCTGGTGAAAAAGAATGAGGAAGCTGCTGATGAGGTCTTCAA
AGACATCATGGAGTCCAGCCCGGGCTCCAGCCCGCCCAACCTGACTCCAAAACCCCTCCGGCGGCAGGTCACCGTGGCCCCTGCCTCGGG
CCTCCCCCACAAGGAAGAAGCTGGAAAGGGCAGTGCCTTAGGGACCCCTGCTGCAGCTGAGCCAGTGACCCCCACCAGCAAAGCAGGCTC
AGGTGCACCAGGGGGCACCAGCAAGGGCCCCGCCGAGGAGTCCAGAGTGAGGAGGCACAAGCACTCCTCTGAGTCGCCAGGGAGGGACAA
GGGGAAATTGTCCAGGCTCAAACCTGCCCCGCCGCCCCCACCAGCAGCCTCTGCAGGGAAGGCTGGAGGAAAGCCCTCGCAGAGCCCGAG
CCAGGAGGCGGCCGGGGAGGCAGTCCTGGGCGCAAAGACAAAAGCCACGAGTCTGGTTGATGCTGTGAACAGTGACGCTGCCAAGCCCAG
CCAGCCGGGAGAGGGCCTCAAAAAGCCCGTGCTCCCGGCCACTCCAAAGCCACAGTCCGCCAAGCCGTCGGGGACCCCCATCAGCCCAGC
CCCCGTTCCCTCCACGTTGCCATCAGCATCCTCGGCCCTGGCAGGGGACCAGCCGTCTTCCACCGCCTTCATCCCTCTCATATCAACCCG
AGTGTCTCTTCGGAAAACCCGCCAGCCTCCAGAGCGGATCGCCAGCGGCGCCATCACCAAGGGCGTGGTCCTGGACAGCACCGAGGCGCT
GTGCCTCGCCATCTCTAGGAACTCCGAGCAGATGGCCAGCCACAGCGCAGTGCTGGAGGCCGGCAAAAACCTCTACACGTTCTGCGTGAG
CTATGTGGATTCCATCCAGCAAATGAGGAACAAGTTTGCCTTCCGAGAGGCCATCAACAAACTGGAGAATAATCTCCGGGAGCTTCAGAT
CTGCCCGGCGACAGCAGGCAGTGGTCCAGCGGCCACTCAGGACTTCAGCAAGCTCCTCAGTTCGGTGAAGGAAATCAGTGACATAGTGCA
GAGGTAGCAGCAGTCAGGGGTCAGGTGTCAGGCCCGTCGGAGCTGCCTGCAGCACATGCGGGCTCGCCCATACCCGTGACAGTGGCTGAC
AAGGGACTAGTGAGTCAGCACCTTGGCCCAGGAGCTCTGCGCCAGGCAGAGCTGAGGGCCCTGTGGAGTCCAGCTCTACTACCTACGTTT
GCACCGCCTGCCCTCCCGCACCTTCCTCCTCCCCGCTCCGTCTCTGTCCTCGAATTTTATCTGTGGAGTTCCTGCTCCGTGGACTGCAGT
CGGCATGCCAGGACCCGCCAGCCCCGCTCCCACCTAGTGCCCCAGACTGAGCTCTCCAGGCCAGGTGGGAACGGCTGATGTGGACTGTCT
TTTTCATTTTTTTCTCTCTGGAGCCCCTCCTCCCCCGGCTGGGCCTCCTTCTTCCACTTCTCCAAGAATGGAAGCCTGAACTGAGGCCTT
GTGTGTCAGGCCCTCTGCCTGCACTCCCTGGCCTTGCCCGTCGTGTGCTGAAGACATGTTTCAAGAACCGCATTTCGGGAAGGGCATGCA
CGGGCATGCACACGGCTGGTCACTCTGCCCTCTGCTGCTGCCCGGGGTGGGGTGCACTCGCCATTTCCTCACGTGCAGGACAGCTCTTGA
TTTGGGTGGAAAACAGGGTGCTAAAGCCAACCAGCCTTTGGGTCCTGGGCAGGTGGGAGCTGAAAAGGATCGAGGCATGGGGCATGTCCT
TTCCATCTGTCCACATCCCCAGAGCCCAGCTCTTGCTCTCTTGTGACGTGCACTGTGAATCCTGGCAAGAAAGCTTGAGTCTCAAGGGTG
GCAGGTCACTGTCACTGCCGACATCCCTCCCCCAGCAGAATGGAGGCAGGGGACAAGGGAGGCAGTGGCTAGTGGGGTGAACAGCTGGTG
CCAAATAGCCCCAGACTGGGCCCAGGCAGGTCTGCAAGGGCCCAGAGTGAACCGTCCTTTCACACATCTGGGTGCCCTGAAAGGGCCCTT
CCCCTCCCCCACTCCTCTAAGACAAAGTAGATTCTTACAAGGCCCTTTCCTTTGGAACAAGACAGCCTTCACTTTTCTGAGTTCTTGAAG
CATTTCAAAGCCCTGCCTCTGTGTAGCCGCCCTGAGAGAGAATAGAGCTGCCACTGGGCACCTGCGCACAGGTGGGAGGAAAGGGCCTGG
CCAGTCCTGGTCCTGGCTGCACTCTTGAACTGGGCGAATGTCTTATTTAATTACCGTGAGTGACATAGCCTCATGTTCTGTGGGGGTCAT
CAGGGAGGGTTAGGAAAACCACAAACGGAGCCCCTGAAAGCCTCACGTATTTCACAGAGCACGCCTGCCATCTTCTCCCCGAGGCTGCCC
CAGGCCGGAGCCCAGATACGGGGGCTGTGACTCTGGGCAGGGACCCGGGGTCTCCTGGACCTTGACAGAGCAGCTAACTCCGAGAGCAGT
GGGCAGGTGGCCGCCCCTGAGGCTTCACGCCGGGAGAAGCCACCTTCCCACCCCTTCATACCGCCTCGTGCCAGCAGCCTCGCACAGGCC
CTAGCTTTACGCTCATCACCTAAACTTGTACTTTATTTTTCTGATAGAAATGGTTTCCTCTGGATCGTTTTATGCGGTTCTTACAGCACA
TCACCTCTTTGCCCCCGACGGCTGTGACGCAGCCGGAGGGAGGCACTAGTCACCGACAGCGGCCTTGAAGACAGAGCAAAGCGCCCACCC
AGGTCCCCCGACTGCCTGTCTCCATGAGGTACTGGTCCCTTCCTTTTGTTAACGTGATGTGCCACTATATTTTACACGTATCTCTTGGTA
TGCATCTTTTATAGACGCTCTTTTCTAAGTGGCGTGTGCATAGCGTCCTGCCCTGCCCCCTCGGGGGCCTGTGGTGGCTCCCCCTCTGCT
TCTCGGGGTCCAGTGCATTTTGTTTCTGTATATGATTCTCTGTGGTTTTTTTTGAATCCAAATCTGTCCTCTGTAGTATTTTTTAAATAA
ATCAGTGTTTACATTAGAA""".replace("\n", "").replace(" ", "")

bp_index = 4073
# 1. Výpočet maximálního symetrického okna
vzdalenost_k_zacatku = bp_index
vzdalenost_ke_konci = len(full_sequence) - bp_index

# Základní teoretické maximum, aby byl zlom uprostřed
max_mozne = min(vzdalenost_k_zacatku, vzdalenost_ke_konci) * 2

# Úprava na nejbližší nižší číslo dělitelné 12
max_symetricke_12 = (max_mozne // 12) * 12

print(f"Tvůj zlom: {bp_index}")
print(f"Maximální okno (symetrické a dělitelné 12): {max_symetricke_12} bp")

# 2. Generování oken (dvojnásobky)
window_sizes = []
# Začínáme na 120 (120 / 12 = 10 -> Perfektní)
aktualni_velikost = 120 

while aktualni_velikost < max_symetricke_12:
    window_sizes.append(aktualni_velikost)
    aktualni_velikost *= 2

# 3. Přidání toho největšího okna (dělitelného 12)
if max_symetricke_12 not in window_sizes and max_symetricke_12 >= 120:
    window_sizes.append(max_symetricke_12)

window_sizes.sort()

# Kontrola
print(f"Vygenerovaná okna: {window_sizes}")
print(f"Max okno dělitelné 12: {max_symetricke_12 % 12 == 0}")

Tvůj zlom: 4073
Maximální okno (symetrické a dělitelné 12): 8136 bp
Vygenerovaná okna: [120, 240, 480, 960, 1920, 3840, 7680, 8136]
Max okno dělitelné 12: True


In [18]:
#Výpočet
results = []

print(f"Spouštím výpočet na {device}...")

for size in window_sizes:
    # 1. Výřez sekvence
    half = size // 2
    start = max(0, bp_index - half)
    end = min(len(full_sequence), bp_index + half)
    current_seq = full_sequence[start:end]
    actual_bp_in_snippet = bp_index - start
    
    # 2. Tokenizace
    inputs = tokenizer(current_seq, return_tensors="pt", truncation=True, max_length=size)
    
    # 3. Přesun na GPU
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        # Žádný autocast, čistý float32 výpočet
        outputs = model(**inputs, output_hidden_states=True)
        
        # Získání embeddingů z poslední vrstvy
        last_hidden_state = outputs.hidden_states[-1]
        
        # Výpočet indexu tokenu
        token_idx = min((actual_bp_in_snippet // 6) + 1, last_hidden_state.shape[1] - 1)
        
        # Stažení do CPU
        bp_embedding = last_hidden_state[0, token_idx, :].detach().cpu()
        
        results.append({
            "size": len(current_seq),
            "embedding": bp_embedding
        })
        
    print(f"Zpracováno okno {size} bp (VRAM: {torch.cuda.memory_allocated() / 1024**2:.0f} MB)")

print("\nVýpočty dokončeny.")

Spouštím výpočet na cuda...
Zpracováno okno 120 bp (VRAM: 1912 MB)
Zpracováno okno 1200 bp (VRAM: 1939 MB)
Zpracováno okno 2400 bp (VRAM: 1971 MB)
Zpracováno okno 3600 bp (VRAM: 1999 MB)
Zpracováno okno 4800 bp (VRAM: 2029 MB)
Zpracováno okno 6000 bp (VRAM: 2057 MB)
Zpracováno okno 7200 bp (VRAM: 2086 MB)
Zpracováno okno 8136 bp (VRAM: 2111 MB)

Výpočty dokončeny.


In [19]:
import torch
import pandas as pd
from torch.nn.functional import cosine_similarity

final_results = []

# 1. Získání referenčního embeddingu (z největšího okna)
# Pojistka: ujistíme se, že je to ve float32 a na CPU
ref_emb = results[-1]["embedding"].detach().cpu().float()

print(f"Zpracovávám {len(results)} výsledků do tabulky...")

for item in results:
    # 2. Příprava aktuálního embeddingu
    curr_emb = item["embedding"].detach().cpu().float()
    
    # 3. Výpočet kosinové podobnosti
    # unsqueeze(0) vytvoří z vektoru matici o jednom řádku, co torch vyžaduje
    sim = cosine_similarity(curr_emb.unsqueeze(0), ref_emb.unsqueeze(0)).item()
    
    # 4. Příprava řádku pro CSV
    row = {
        "Délka": item["size"],
        "Podobnost_s_maximem": sim
    }
    
    # 5. Rozklad embeddingu (1280 čísel) do sloupců
    # Převod na numpy list, aby to pandas snadno schroustal
    emb_list = curr_emb.numpy().tolist()
    for i, val in enumerate(emb_list):
        row[f"emb_{i}"] = val
        
    final_results.append(row)

# 6. Vytvoření DataFrame a uložení
df = pd.DataFrame(final_results)

# Uložení do CSV
output_filename = "analyzaBPembegging9379V2.csv"
df.to_csv(output_filename, index=False)

print("--- VŠE ÚSPĚŠNĚ DOKONČENO ---")
print(f"Výsledná tabulka má rozměry: {df.shape}")
print("\nTabulka bez embedding sloupců:")
print(df[["Délka", "Podobnost_s_maximem"]].to_string(index=False))
print(f"\nKompletní data uložena do: {output_filename}")

Zpracovávám 8 výsledků do tabulky...
--- VŠE ÚSPĚŠNĚ DOKONČENO ---
Výsledná tabulka má rozměry: (8, 1026)

Tabulka bez embedding sloupců:
 Délka  Podobnost_s_maximem
   120             0.390208
  1200             0.626564
  2400             0.966596
  3600             0.983185
  4800             0.991673
  6000             0.993621
  7200             0.997116
  8136             1.000000

Kompletní data uložena do: analyzaBPembegging9379V2.csv
